<a href="https://colab.research.google.com/github/rafaellopesdesa/nsbi-lhc-toolkit/blob/ml4hep_school_tutorial/workshops/ml4hep_tifr_colab/Exercise_9b_SBIBM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Exercise 9b — the multiclass hybrid campaign on all `sbibm` tasks

This notebook takes the best three-class construction from Exercise 9 (multiclass) to the ten tasks in the `sbibm` benchmark.  For every cross-fit fold it trains equal-weight mixtures of LU-mixed conditional spline flows and one structured multiclass ensemble.  The fixed objective is

$$
\mathcal L = \mathcal L_{\rm CE}
  +\lambda_a\left(\mathcal L_{a,\mathrm{direct}}-\log 2\right)
  +\lambda_Z\mathcal L_{\rm normalization}^{\rm cross}
  +\lambda_B\mathcal L_{\rm bridge}^{\rm raw}.
$$

The official reference posterior is hidden until evaluation.  Each completed task writes posterior and posterior-predictive metrics.  The headline aggregate follows the two-part logic of JANA Figure 5: posterior MMD in the first panel and joint posterior-predictive MMD in the second.  Posterior and predictive C2ST remain supplemental diagnostics rather than substitutes for the second MMD panel.

**This is a campaign notebook, not a promise that ten paper-scale fits belong in one Colab session.**  The default runs one selected task.  Choose the task with `EX9B_TASK`, opt into all ten with `EX9B_RUN_ALL=1` only on suitable infrastructure, and set each independent campaign seed with `EX9B_SEED`.  Checkpoints, simulation banks, results, failure manifests, posterior/predictive sample files, and figure scripts are isolated by the resulting seed-bearing run tag.


In [ ]:
# Google Colab setup -- safe to rerun and a no-op outside Colab.
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/rafaellopesdesa/nsbi-lhc-toolkit.git"
BRANCH = "ml4hep_school_tutorial"
USE_DRIVE = True

def run(*args, env=None):
    subprocess.run([str(arg) for arg in args], check=True, env=env)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        ROOT = Path("/content/drive/MyDrive/Colab Notebooks/ml4hep_tifr_colab")
    else:
        ROOT = Path("/content")
    ROOT.mkdir(parents=True, exist_ok=True)
    REPO_DIR = ROOT / "nsbi-lhc-toolkit"
    TUTORIAL_DIR = REPO_DIR / "workshops" / "ml4hep_tifr_colab"
    WORK_DIR = REPO_DIR / "workshops" / "ml4hep_tifr"
    if not (REPO_DIR / ".git").is_dir():
        clone_env = os.environ.copy()
        clone_env["GIT_LFS_SKIP_SMUDGE"] = "1"
        run(
            "git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
            "--branch", BRANCH, REPO_URL, REPO_DIR, env=clone_env,
        )
    else:
        run("git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL)
        run("git", "-C", REPO_DIR, "fetch", "origin", BRANCH)
        run("git", "-C", REPO_DIR, "checkout", BRANCH)
        run("git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH)
    run(
        "git", "-C", REPO_DIR, "sparse-checkout", "set",
        "src", "workshops/ml4hep_tifr_colab",
    )
    for import_dir in (REPO_DIR / "src", TUTORIAL_DIR):
        path = str(import_dir.resolve())
        if path not in sys.path:
            sys.path.insert(0, path)
    run(sys.executable, "-m", "pip", "install", "-q", "nflows==0.14", "pyro-ppl")
    run(sys.executable, "-m", "pip", "install", "-q", "--no-deps", "sbibm==1.1.0")
    WORK_DIR.mkdir(parents=True, exist_ok=True)
    os.chdir(WORK_DIR)
else:
    for candidate in [Path.cwd(), Path.cwd() / "workshops" / "ml4hep_tifr_colab"]:
        if (candidate / "utils_hnpe.py").exists():
            sys.path.insert(0, str(candidate.resolve()))
            break
print("Working directory:", Path.cwd())


In [ ]:
import copy
import gc
import hashlib
import inspect
import json
import math
import os
import random
import traceback
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sbibm
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
from scipy.special import logsumexp
from scipy.spatial.distance import cdist, pdist
from torch.utils.data import DataLoader, TensorDataset

from utils_benchmark import (
    infer_parameter_transform,
    load_or_simulate_bank,
    split_simulation_bank,
    task_recommendation_table,
)
from utils_hnpe import (
    sample_spline_flow_ensemble,
    spline_flow_ensemble_log_prob,
    train_spline_flow_ensemble,
)
from utils_plotting import export_standalone_figure_script

DEFAULT_SEED = 29082026
# The largest legacy-NumPy/sklearn seed is task 9, joint C2ST, observation 10.
MAX_LEGACY_SEED_OFFSET = 100_000 * 9 + 80_000 + 10
MAX_BASE_SEED = 2**32 - 1 - MAX_LEGACY_SEED_OFFSET
SEED = int(os.environ.get("EX9B_SEED", str(DEFAULT_SEED)))
if not 0 <= SEED <= MAX_BASE_SEED:
    raise ValueError(
        f"EX9B_SEED must lie in [0, {MAX_BASE_SEED}] so every derived "
        "legacy NumPy/sklearn seed remains valid."
    )
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def seed_everything(seed):
    random.seed(int(seed))
    np.random.seed(int(seed) % 2**32)
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))

seed_everything(SEED)
print("sbibm version:", sbibm.__version__)
print("Using device:", device)


## 1. The structured three-class experiment

In latent parameter coordinates $z=T(\theta)$, the three equally weighted class laws are

$$
S=\rho_z(z)p(x\mid z),\qquad
P=m(x)q_\phi(z\mid x),\qquad
L=\rho_z(z)q_\eta(x\mid z).
$$

Two ratio heads are sufficient:

$$a(z,x)=\log(S/P),\qquad c(z,x)=\log(S/L),$$

and we fix the class logits to $[a,0,a-c]$.  Thus the pointwise ratio cycle is architectural, not a separately testable relation.  The $a$ head also receives a direct balanced $S/P$ binary loss.  Both $a$ and $c$ have the same enlarged pre-normalized residual architecture (512 hidden features and six residual blocks in FAST/FULL), and classifier members are combined by averaging $a$ and $c$ in log-ratio space.  This geometric ensemble preserves the structured logits exactly.  Every classifier input uses the Exercise-9 robust transform $\operatorname{asinh}[(y-\operatorname{median}y)/(1.4826\,\operatorname{MAD}y)]$, with ordinary standard deviation only as a zero-MAD fallback.  The fitted transform is frozen and shared by class, constraint, validation, and prediction paths.

The frozen references are genuine equal-weight density mixtures,

$$q_\phi=\frac1M\sum_mq_{\phi m},\qquad q_\eta=\frac1M\sum_mq_{\eta m},$$

not one arbitrarily selected flow.  Vector flows use learned LU mixing after each coupling.  A cross-fit keeps every classifier row outside the flow-training rows.  Held-out normalization and bridge diagnostics are recomputed from the deployed averaged $a/c$ heads; averages of member-level diagnostics are saved only as supplemental columns.


## 2. Why normalization and the raw-evidence bridge are different

Conditional normalization requires

$$Z_P(x)=\mathbb E_{q_\phi(z\mid x)}e^{a(z,x)}=1,\qquad
  Z_L(z)=\mathbb E_{q_\eta(x\mid z)}e^{c(z,x)}=1.$$

For each anchor we make independent Monte Carlo estimates $Z_A,Z_B$ and optimize $(Z_A-1)(Z_B-1)$.  Its expectation is $(Z-1)^2$; unlike squaring one noisy estimate, it does not reward Monte Carlo variance.  A pooled positive square is retained for validation and plotting.

Normalization does not impose Bayes compatibility.  The **raw evidence path** is

$$
\log m_{\rm raw}(x;z)=
   \log\rho_z(z)+\log q_\eta(x\mid z)-\log q_\phi(z\mid x)
   +c(z,x)-a(z,x).
$$

It must be independent of $z$ at fixed $x$, so

$$\mathcal L_{\rm bridge}^{\rm raw}
  =\mathbb E_x\operatorname{Var}_{z\sim q_\phi(\cdot\mid x)}
   [\log m_{\rm raw}(x;z)].$$

“Raw” means that no noisy $\widehat Z_P$ or $\widehat Z_L$ is inserted inside this variance.  The cross-normalization terms train those partitions separately.  The bridge base uses only the learned flow-mixture densities and the known prior density; it never uses the simulator likelihood or a reference posterior.  Gradients update the classifier heads, while the flow mixtures remain frozen in this exercise.  Bridge candidates use IID categorical flow-member labels before the ordinary unbiased sample variance is formed.  Balanced member allocation remains useful lower-noise quadrature for class generation and normalization, but would invalidate the IID variance estimator.

The same two learned objects define a corrected generative likelihood, $\widehat p(x\mid z)\propto q_\eta(x\mid z)e^{c(z,x)}$.  For posterior-predictive generation we draw a finite candidate set from $q_\eta$, weight it by $e^c$, and sample-importance-resample (SIR) one $x$ per posterior $z$.  This needs no simulator and remains paired with the same fold-local classifier and flow mixture.

Training is staged: CE/direct-$a$ first, cross-normalization next, and the bridge only after the conditional masses have begun to settle.  Checkpoint selection uses the held-out A/B cross-mass estimate, exactly matching the trained normalization objective.  The non-negative pooled square is a lower-variance monitoring curve only and is never substituted into the checkpoint objective.  Best-checkpoint selection and early stopping are disabled until both constraint ramps have reached full strength, so a pre-ramp model cannot win.  After that gate, the signed A/B estimator enters the objective exactly; its finite-sample fluctuations, including negative values, must be inspected rather than interpreted as a non-negative discrepancy.


In [ ]:
ALL_TASKS = (
    "gaussian_linear", "gaussian_linear_uniform", "gaussian_mixture",
    "two_moons", "slcp", "slcp_distractors", "bernoulli_glm",
    "bernoulli_glm_raw", "sir", "lotka_volterra",
)
PROFILE = os.environ.get("EX9B_PROFILE", "FAST").upper()  # SMOKE, FAST, FULL
TASK_NAME = os.environ.get("EX9B_TASK", "two_moons")
RUN_ALL_TASKS = os.environ.get("EX9B_RUN_ALL", "0") == "1"
LOAD_IF_AVAILABLE = os.environ.get("EX9B_LOAD", "1") == "1"
FAIL_ON_TASK_ERROR = os.environ.get("EX9B_FAIL_ON_ERROR", "1") == "1"

PROFILES = {
    "SMOKE": dict(
        num_simulations=1_000, n_folds=2, flow_members=1,
        flow_epochs=2, class_epochs=3, class_width=128, class_blocks=2,
        norm_groups=32, norm_inner=4, bridge_groups=24, bridge_inner=4,
        constraint_batch=8, n_proposal=1_500, n_posterior=1_000,
        observations=[1], observation_jitters=4,
        predictive_candidates=4, predictive_samples=256,
        predictive_reference_calls=256,
    ),
    "FAST": dict(
        num_simulations=10_000, n_folds=2, flow_members=2,
        flow_epochs=25, class_epochs=40, class_width=512, class_blocks=6,
        norm_groups=256, norm_inner=16, bridge_groups=128, bridge_inner=12,
        constraint_batch=24, n_proposal=30_000, n_posterior=5_000,
        observations=[1, 2, 3], observation_jitters=8,
        predictive_candidates=8, predictive_samples=1_000,
        predictive_reference_calls=1_000,
    ),
    "FULL": dict(
        num_simulations=100_000, n_folds=5, flow_members=4,
        flow_epochs=90, class_epochs=120, class_width=512, class_blocks=6,
        norm_groups=1_024, norm_inner=64, bridge_groups=512, bridge_inner=32,
        constraint_batch=32, n_proposal=150_000, n_posterior=10_000,
        observations=list(range(1, 11)), observation_jitters=32,
        predictive_candidates=16, predictive_samples=2_000,
        predictive_reference_calls=2_000,
    ),
}
if PROFILE not in PROFILES:
    raise ValueError(f"PROFILE must be one of {tuple(PROFILES)}")
if TASK_NAME not in ALL_TASKS:
    raise ValueError(f"TASK_NAME must be one of {ALL_TASKS}")
campaign = PROFILES[PROFILE]
TASKS_TO_RUN = ALL_TASKS if RUN_ALL_TASKS else (TASK_NAME,)

LAMBDA_DIRECT = 1.0
LAMBDA_NORMALIZATION = 0.15
LAMBDA_BRIDGE = 0.03
LOG_RATIO_BOUND = 18.0
CAMPAIGN_SCHEMA = "structured_v7_seeded_deployed_diagnostics_all_observations"
CAMPAIGN_SIGNATURE = "sha256-" + hashlib.sha256(json.dumps({
    "campaign_schema": CAMPAIGN_SCHEMA,
    "profile": PROFILE, "campaign": campaign,
    "lambda_direct": LAMBDA_DIRECT,
    "lambda_normalization": LAMBDA_NORMALIZATION,
    "lambda_bridge": LAMBDA_BRIDGE,
    "log_ratio_bound": LOG_RATIO_BOUND,
    "flow_topology": "equal_weight_lu_rqs_ensemble_v1",
    "classifier_transform": "robust_median_mad_asinh_v1",
    "bridge_sampling": "iid_unbiased_raw_evidence_v1",
    "data_semantics": "bernoulli_sir_dequant_lv_lognormal_continuous_v1",
    "posterior_predictive": "fold_local_qeta_c_finite_k_sir_v1",
}, sort_keys=True).encode("utf-8")).hexdigest()[:12]

def campaign_run_tag(seed):
    return (
        f"v7_{PROFILE.lower()}_seed{int(seed)}_n{campaign['num_simulations']}_"
        f"k{campaign['n_folds']}_flowmix{campaign['flow_members']}_"
        f"cfg{CAMPAIGN_SIGNATURE}_structured_ce_norm_bridge_lvcontinuous"
    )

RUN_TAG = campaign_run_tag(SEED)
aggregate_seed_text = os.environ.get("EX9B_AGGREGATE_SEEDS", "").strip()
if aggregate_seed_text:
    try:
        AGGREGATE_SEEDS = tuple(dict.fromkeys(
            int(value.strip()) for value in aggregate_seed_text.split(",")
            if value.strip()
        ))
    except ValueError as exc:
        raise ValueError("EX9B_AGGREGATE_SEEDS must be comma-separated integers.") from exc
else:
    AGGREGATE_SEEDS = (SEED,)
if not AGGREGATE_SEEDS or any(not 0 <= value <= MAX_BASE_SEED for value in AGGREGATE_SEEDS):
    raise ValueError(
        f"Every aggregate seed must lie in [0, {MAX_BASE_SEED}] so its "
        "derived legacy NumPy/sklearn seeds remain valid."
    )
AGGREGATE_RUN_TAGS = tuple(campaign_run_tag(seed) for seed in AGGREGATE_SEEDS)
AGGREGATE_TAG = (
    RUN_TAG if AGGREGATE_SEEDS == (SEED,) else
    f"v7_{PROFILE.lower()}_seeds{'-'.join(map(str, AGGREGATE_SEEDS))}_"
    f"cfg{CAMPAIGN_SIGNATURE}_aggregate"
)
MODEL_ROOT = Path("models_exercise9b_sbibm")
CACHE_ROOT = Path("dataframes_exercise9b_sbibm")
RESULT_ROOT = Path("exercise9b_sbibm_results")
FIGURE_ROOT = Path("exercise9b_figures_scripts")
for directory in (MODEL_ROOT, CACHE_ROOT, RESULT_ROOT, FIGURE_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

recommendations = task_recommendation_table().copy()
lv_row = recommendations["task"] == "lotka_volterra"
recommendations.loc[lv_row, "data_type"] = "continuous LogNormal-noised trajectories"
recommendations.loc[lv_row, "reason"] = (
    "The official simulator returns continuous LogNormal-noised trajectories; "
    "no dequantization is applied."
)
display(recommendations.style.hide(axis="index"))
print(json.dumps({
    "profile": PROFILE, "seed": SEED,
    "tasks_this_run": TASKS_TO_RUN, "run_tag": RUN_TAG,
    "campaign_signature": CAMPAIGN_SIGNATURE,
    "aggregate_seeds": AGGREGATE_SEEDS,
    "aggregate_run_tags": AGGREGATE_RUN_TAGS,
    "loss": "CE + direct-a + cross-normalization + raw-evidence-bridge",
    "flow_mixture_members": campaign["flow_members"],
    "classifier_members": campaign["flow_members"],
    "crossfit_folds": campaign["n_folds"],
    "predictive_candidates_per_theta": campaign["predictive_candidates"],
    "predictive_samples": campaign["predictive_samples"],
    "evaluation_simulator_calls_per_observation": campaign["predictive_reference_calls"],
}, indent=2))


## 3. Discrete observations: an explicit change of measure

`bernoulli_glm_raw` and SIR live on an integer lattice.  We use randomized dequantization $\widetilde x=x+u$ with $u_j\sim\mathrm{Uniform}[-w_j/2,w_j/2]$ and train a **continuous density of $\widetilde x$**.  For raw binary/count data $w_j=1$, so integrating a perfect dequantized density over the unit cell recovers a probability mass.  At an observed integer vector, posterior evaluation averages over multiple cell jitters rather than pretending the cell center is a continuous observation.

`bernoulli_glm` is a derived sufficient statistic on an irregular finite support.  Its data-driven rectangular jitter is only a declared smoothing convention; it is **not** an exact PMF representation.  Accordingly, density/evidence values from that task must not be labeled exact likelihoods.  C2ST and MMD against the official posterior remain meaningful tests of the resulting inference procedure.

Official `sbibm` Lotka--Volterra observations are continuous LogNormal-noised trajectories.  They therefore receive zero dequantization width and no observation jitter in training, posterior evaluation, or predictive evaluation.  The official SIR and Lotka--Volterra simulators may require the Julia/`diffeqtorch` stack.  An exact cached prior-predictive bank avoids repeating the **training** simulations, but it does not replace the official task import, reference assets, or fresh simulator calls required for posterior-predictive evaluation.  This notebook neither silently drops those stages nor substitutes a different simulator.  A failure writes a JSON manifest and, by default, raises after displaying it.


In [ ]:
DISCRETE_TASKS = {"bernoulli_glm", "bernoulli_glm_raw", "sir"}

def require_finite_rows(task_name, stage, values):
    if torch.is_tensor(values):
        values = values.detach().cpu().numpy()
    array = np.asarray(values)
    if array.ndim == 0:
        array = array.reshape(1, 1)
    elif array.ndim == 1:
        array = array.reshape(-1, 1)
    flat = array.reshape(len(array), -1)
    finite_rows = np.isfinite(flat).all(axis=1)
    if not finite_rows.all():
        bad = int((~finite_rows).sum())
        raise FloatingPointError(
            f"{task_name} [{stage}]: {bad}/{len(flat)} rows contain non-finite values"
        )
    return array

def dequantization_widths(task_name, values):
    values = np.asarray(values, dtype=np.float32)
    if task_name not in DISCRETE_TASKS:
        return np.zeros(values.shape[1], dtype=np.float32)
    if task_name != "bernoulli_glm":
        return np.ones(values.shape[1], dtype=np.float32)
    # The sufficient-statistic support is irregular.  This deterministic
    # local scale defines smoothing only; it does not claim PMF semantics.
    widths = []
    for column in values.T:
        unique = np.unique(np.round(column[: min(len(column), 20_000)], 7))
        gaps = np.diff(unique)
        gaps = gaps[gaps > 1.0e-7]
        robust_scale = np.std(column, dtype=np.float64)
        width = np.median(gaps) if len(gaps) else 0.02 * robust_scale
        width = np.clip(width, 1.0e-4, max(1.0e-4, 0.10 * robust_scale))
        widths.append(width)
    return np.asarray(widths, dtype=np.float32)

def dequantize(values, widths, seed):
    values = np.asarray(values, dtype=np.float32)
    widths = np.asarray(widths, dtype=np.float32)
    if not np.any(widths):
        return values.copy()
    rng = np.random.default_rng(int(seed))
    return (values + (rng.random(values.shape) - 0.5) * widths).astype(np.float32)

def install_inverse_rqs_float64_retry():
    """Retry the same finite inverse-RQS tensors in float64 on cancellation."""
    import functools
    import importlib
    module = importlib.import_module("nflows.transforms.splines.rational_quadratic")
    original = module.rational_quadratic_spline
    if getattr(original, "_ex9b_float64_retry", False):
        return original
    signature = inspect.signature(original)

    @functools.wraps(original)
    def guarded(*args, **kwargs):
        try:
            return original(*args, **kwargs)
        except AssertionError as error32:
            bound = signature.bind(*args, **kwargs)
            inputs = bound.arguments["inputs"]
            inverse = bound.arguments.get("inverse", False)
            if not inverse or inputs.dtype != torch.float32:
                raise
            floating = [v for v in (*args, *kwargs.values()) if torch.is_tensor(v) and v.is_floating_point()]
            if any(not bool(torch.isfinite(v).all()) for v in floating):
                raise FloatingPointError("Non-finite tensor reached inverse RQS") from error32
            convert = lambda v: v.double() if torch.is_tensor(v) and v.is_floating_point() else v
            try:
                output, logdet = original(
                    *(convert(v) for v in args),
                    **{k: convert(v) for k, v in kwargs.items()},
                )
            except AssertionError as error64:
                raise RuntimeError("Inverse RQS discriminant also failed in float64") from error64
            if not bool(torch.isfinite(output).all() and torch.isfinite(logdet).all()):
                raise FloatingPointError("Float64 inverse RQS returned non-finite values")
            guarded.retry_count += 1
            if guarded.retry_count == 1:
                warnings.warn("Inverse-RQS cancellation: retrying identical tensors in float64.", RuntimeWarning)
            return output.to(inputs.dtype), logdet.to(inputs.dtype)

    guarded._ex9b_float64_retry = True
    guarded.retry_count = 0
    module.rational_quadratic_spline = guarded
    importlib.import_module("nflows.transforms.splines").rational_quadratic_spline = guarded
    importlib.import_module("nflows.transforms.autoregressive").rational_quadratic_spline = guarded
    return guarded

RQS_GUARD = install_inverse_rqs_float64_retry()


In [ ]:
class PreNormResidualBlock(nn.Module):
    def __init__(self, width):
        super().__init__()
        self.norm = nn.LayerNorm(int(width))
        self.linear_one = nn.Linear(int(width), int(width))
        self.linear_two = nn.Linear(int(width), int(width))

    def forward(self, values):
        update = self.linear_one(self.norm(values))
        update = self.linear_two(F.silu(update))
        return values + update / math.sqrt(2.0)

class RatioHead(nn.Module):
    def __init__(self, input_dim, width, blocks):
        super().__init__()
        self.input_layer = nn.Linear(int(input_dim), int(width))
        self.blocks = nn.ModuleList([PreNormResidualBlock(width) for _ in range(int(blocks))])
        self.final_norm = nn.LayerNorm(int(width))
        self.output_layer = nn.Linear(int(width), 1)
        nn.init.zeros_(self.output_layer.weight)
        nn.init.zeros_(self.output_layer.bias)

    def forward(self, values):
        hidden = F.silu(self.input_layer(values))
        for block in self.blocks:
            hidden = block(hidden)
        return self.output_layer(F.silu(self.final_norm(hidden)))[:, 0]

class StructuredThreeClass(nn.Module):
    def __init__(self, input_dim, width, blocks, log_ratio_bound=18.0):
        super().__init__()
        self.a_head = RatioHead(input_dim, width, blocks)
        self.c_head = RatioHead(input_dim, width, blocks)
        self.log_ratio_bound = float(log_ratio_bound)

    def heads(self, values):
        bound = self.log_ratio_bound
        a = self.a_head(values)
        c = self.c_head(values)
        if bound > 0:
            a = bound * torch.tanh(a / bound)
            c = bound * torch.tanh(c / bound)
        return a, c

    def forward(self, values):
        a, c = self.heads(values)
        return torch.stack([a, torch.zeros_like(a), a - c], dim=1)

def model_parameter_counts(model):
    return {
        "a_head": sum(p.numel() for p in model.a_head.parameters()),
        "c_head": sum(p.numel() for p in model.c_head.parameters()),
        "total": sum(p.numel() for p in model.parameters()),
    }

preview_model = StructuredThreeClass(
    12, campaign["class_width"], campaign["class_blocks"], LOG_RATIO_BOUND
)
print("Structured classifier parameter counts:", model_parameter_counts(preview_model))
del preview_model


In [ ]:
def flow_model_config(n_target):
    # The 100-dimensional distractor/raw tasks need a memory-aware version
    # of the same LU-mixed topology; this is recorded in every checkpoint.
    high_dimensional = int(n_target) > 20
    return {
        "n_unbounded_affine_layers": 2,
        "affine_hidden_features": 256 if high_dimensional else 512,
        "affine_hidden_layers": 3,
        "use_layer_permutations": False,
        "n_coupling_layers": 8 if high_dimensional else 10,
        "hidden_features": 384 if high_dimensional else 512,
        "hidden_layers": 4,
        "spline_num_bins": 16 if high_dimensional else 32,
        "spline_tail_bound": 15.0,
        "dropout_probability": 0.0,
        "activation": "silu",
        "identity_initialization": True,
        "linear_mixing": "lu",
        "standardize_target": True,
    }

FLOW_TRAINING_CONFIG = {
    "batch_size": 1024,
    "n_epochs": campaign["flow_epochs"],
    "learning_rate": 1.0e-3,
    "lr_scheduler_factor": 0.3,
    "lr_scheduler_patience": 4,
    "min_learning_rate": 1.0e-6,
    "validation_fraction": 0.15,
    "patience": max(4, campaign["flow_epochs"] // 4),
    "gradient_clip": 5.0,
}

def draw_flow_mixture(flow_packs, n_samples, contexts, seed, allocation="balanced"):
    contexts = np.atleast_2d(np.asarray(contexts, dtype=np.float32))
    seed_everything(seed)
    values = sample_spline_flow_ensemble(
        flow_packs, int(n_samples), context=contexts, seed=int(seed),
        batch_size=16_384, allocation=str(allocation),
    )
    values = np.asarray(values, dtype=np.float32)
    n_features = int(np.asarray(flow_packs[0]["target_scaler"].mean).size)
    if len(contexts) == 1:
        values = values.reshape(1, int(n_samples), n_features)
    expected = (len(contexts), int(n_samples), n_features)
    if values.shape != expected:
        raise RuntimeError(f"Unexpected mixture sample shape {values.shape}; expected {expected}")
    if not np.isfinite(values).all():
        raise FloatingPointError("Flow-mixture sampling returned non-finite values")
    return values

def prior_log_prob_latent(task, transform, latent):
    latent = np.asarray(latent, dtype=np.float32)
    theta = transform.inverse(latent)
    with torch.no_grad():
        values = task.get_prior_dist().log_prob(torch.as_tensor(theta, dtype=torch.float32))
    values = values.detach().cpu().numpy()
    if values.ndim > 1:
        values = values.sum(axis=tuple(range(1, values.ndim)))
    return np.asarray(values, dtype=np.float64) + transform.log_abs_det_inverse(latent)

def build_class_groups(z, x, q_phi, q_eta, seed):
    z = np.asarray(z, dtype=np.float32)
    x = np.asarray(x, dtype=np.float32)
    z_p = draw_flow_mixture(q_phi, 1, x, seed + 1)[:, 0, :]
    x_l = draw_flow_mixture(q_eta, 1, z, seed + 2)[:, 0, :]
    simulator = np.column_stack([z, x])
    posterior_reference = np.column_stack([z_p, x])
    likelihood_reference = np.column_stack([z, x_l])
    groups = np.stack([simulator, posterior_reference, likelihood_reference], axis=1)
    if not np.isfinite(groups).all():
        raise FloatingPointError("Non-finite class group")
    return groups.astype(np.float32)


In [ ]:
def _anchor_rows(values, n_rows, rng):
    replace = int(n_rows) > len(values)
    return values[rng.choice(len(values), size=int(n_rows), replace=replace)]

def build_constraint_bundle(task, transform, z, x, q_phi, q_eta, seed):
    input_dim = z.shape[1] + x.shape[1]
    memory_factor = max(1, int(math.ceil(input_dim / 24)))
    norm_groups = max(24, campaign["norm_groups"] // memory_factor)
    norm_inner = max(4, int(campaign["norm_inner"] / math.sqrt(memory_factor)))
    bridge_groups = max(24, campaign["bridge_groups"] // memory_factor)
    bridge_inner = max(4, int(campaign["bridge_inner"] / math.sqrt(memory_factor)))
    rng = np.random.default_rng(int(seed))

    x_anchor = _anchor_rows(x, norm_groups, rng)
    zp_a = draw_flow_mixture(q_phi, norm_inner, x_anchor, seed + 1)
    zp_b = draw_flow_mixture(q_phi, norm_inner, x_anchor, seed + 2)
    x_repeat = np.repeat(x_anchor[:, None, :], norm_inner, axis=1)

    z_anchor = _anchor_rows(z, norm_groups, rng)
    zl_a = draw_flow_mixture(q_eta, norm_inner, z_anchor, seed + 3)
    zl_b = draw_flow_mixture(q_eta, norm_inner, z_anchor, seed + 4)
    z_repeat = np.repeat(z_anchor[:, None, :], norm_inner, axis=1)

    x_bridge = _anchor_rows(x, bridge_groups, rng)
    z_bridge = draw_flow_mixture(
        q_phi, bridge_inner, x_bridge, seed + 5, allocation="iid"
    )
    xb_repeat = np.repeat(x_bridge[:, None, :], bridge_inner, axis=1)
    flat_z = z_bridge.reshape(-1, z.shape[1])
    flat_x = xb_repeat.reshape(-1, x.shape[1])
    bridge_base = (
        prior_log_prob_latent(task, transform, flat_z)
        + spline_flow_ensemble_log_prob(q_eta, flat_x, context=flat_z)
        - spline_flow_ensemble_log_prob(q_phi, flat_z, context=flat_x)
    ).reshape(bridge_groups, bridge_inner)

    bundle = {
        "zp_a": np.concatenate([zp_a, x_repeat], axis=2),
        "zp_b": np.concatenate([zp_b, x_repeat], axis=2),
        "zl_a": np.concatenate([z_repeat, zl_a], axis=2),
        "zl_b": np.concatenate([z_repeat, zl_b], axis=2),
        "bridge": np.concatenate([z_bridge, xb_repeat], axis=2),
        "bridge_base": bridge_base.astype(np.float32),
    }
    for name, values in bundle.items():
        if not np.isfinite(values).all():
            raise FloatingPointError(f"Non-finite constraint array {name}")
    return bundle

def fit_classifier_transform(points):
    points = np.asarray(points, dtype=np.float32)
    center = np.median(points, axis=0).astype(np.float32)
    mad = np.median(np.abs(points - center), axis=0).astype(np.float32)
    robust_scale = (1.4826 * mad).astype(np.float32)
    ordinary_scale = points.std(axis=0, dtype=np.float64).astype(np.float32)
    scale = np.where(robust_scale > 1.0e-6, robust_scale, ordinary_scale)
    scale = np.where(scale > 1.0e-6, scale, 1.0).astype(np.float32)
    return center, scale

def transform_classifier_points(points, center, scale):
    points = np.asarray(points, dtype=np.float32)
    return np.arcsinh((points - center) / scale).astype(np.float32)

def prepare_constraint_bundle(bundle, center, scale):
    prepared = {}
    for name, values in bundle.items():
        values = np.asarray(values, dtype=np.float32)
        if name != "bridge_base":
            values = transform_classifier_points(values, center, scale)
        prepared[name] = torch.as_tensor(values, dtype=torch.float32)
    return prepared

def logmeanexp_torch(values, dim):
    return torch.logsumexp(values, dim=dim) - math.log(values.shape[dim])

def constraint_terms(model, bundle, index):
    def heads(name):
        points = bundle[name][index].to(device)
        shape = points.shape[:-1]
        a, c = model.heads(points.reshape(-1, points.shape[-1]))
        return a.reshape(shape), c.reshape(shape)

    a_zp_a, _ = heads("zp_a")
    a_zp_b, _ = heads("zp_b")
    _, c_zl_a = heads("zl_a")
    _, c_zl_b = heads("zl_b")
    log_zp_a = logmeanexp_torch(a_zp_a, 1)
    log_zp_b = logmeanexp_torch(a_zp_b, 1)
    log_zl_a = logmeanexp_torch(c_zl_a, 1)
    log_zl_b = logmeanexp_torch(c_zl_b, 1)
    dzp_a, dzp_b = torch.expm1(log_zp_a.double()), torch.expm1(log_zp_b.double())
    dzl_a, dzl_b = torch.expm1(log_zl_a.double()), torch.expm1(log_zl_b.double())
    cross = 0.5 * ((dzp_a * dzp_b).mean() + (dzl_a * dzl_b).mean())
    monitor = 0.5 * (
        (0.5 * (dzp_a + dzp_b)).square().mean()
        + (0.5 * (dzl_a + dzl_b)).square().mean()
    )
    a_bridge, c_bridge = heads("bridge")
    raw_evidence = bundle["bridge_base"][index].to(device) + c_bridge - a_bridge
    bridge = raw_evidence.var(dim=1, unbiased=True).mean()
    if not bool(torch.isfinite(cross) and torch.isfinite(monitor) and torch.isfinite(bridge)):
        raise FloatingPointError("Non-finite structured constraint loss")
    return cross.float(), monitor.float(), bridge.float()


In [ ]:
def classifier_fingerprint(train_groups, validation_groups, train_bundle, validation_bundle, config):
    digest = hashlib.sha256(json.dumps(config, sort_keys=True).encode("utf-8"))
    for values in [train_groups, validation_groups] + [
        train_bundle[k] for k in sorted(train_bundle)
    ] + [validation_bundle[k] for k in sorted(validation_bundle)]:
        array = np.ascontiguousarray(values)
        digest.update(str(array.shape).encode("ascii"))
        digest.update(str(array.dtype).encode("ascii"))
        digest.update(array.view(np.uint8))
    return digest.hexdigest()

def safe_torch_load(path):
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)

def class_losses(model, groups):
    groups = groups.to(device)
    flat = groups.reshape(-1, groups.shape[-1])
    logits = model(flat).reshape(len(groups), 3, 3)
    labels = torch.arange(3, device=device).repeat(len(groups))
    ce = F.cross_entropy(logits.reshape(-1, 3), labels)
    a, _ = model.heads(flat)
    a = a.reshape(len(groups), 3)
    direct = 0.5 * (
        F.binary_cross_entropy_with_logits(a[:, 0], torch.ones_like(a[:, 0]))
        + F.binary_cross_entropy_with_logits(a[:, 1], torch.zeros_like(a[:, 1]))
    )
    return ce, direct

def validation_objective(model, groups, bundle, chunk_size=512):
    model.eval()
    ce_sum = direct_sum = 0.0
    with torch.no_grad():
        for start in range(0, len(groups), chunk_size):
            batch = groups[start:start + chunk_size]
            ce, direct = class_losses(model, batch)
            weight = len(batch)
            ce_sum += float(ce.cpu()) * weight
            direct_sum += float(direct.cpu()) * weight
        cross_sum = monitor_sum = bridge_sum = count = 0.0
        n_constraint = len(bundle["bridge"])
        # Normalization and bridge group counts may differ. Evaluate them
        # separately by cycling the shorter index without changing arrays.
        n_eval = max(len(bundle["zp_a"]), n_constraint)
        for start in range(0, n_eval, min(128, chunk_size)):
            stop = min(n_eval, start + min(128, chunk_size))
            raw = torch.arange(start, stop)
            index_norm = raw % len(bundle["zp_a"])
            index_bridge = raw % len(bundle["bridge"])
            view = {
                "zp_a": bundle["zp_a"], "zp_b": bundle["zp_b"],
                "zl_a": bundle["zl_a"], "zl_b": bundle["zl_b"],
                "bridge": bundle["bridge"], "bridge_base": bundle["bridge_base"],
            }
            # constraint_terms expects one index. Equalize by a temporary
            # compact view only when the two group counts differ.
            compact = {
                "zp_a": view["zp_a"][index_norm], "zp_b": view["zp_b"][index_norm],
                "zl_a": view["zl_a"][index_norm], "zl_b": view["zl_b"][index_norm],
                "bridge": view["bridge"][index_bridge],
                "bridge_base": view["bridge_base"][index_bridge],
            }
            local = torch.arange(stop - start)
            cross, monitor, bridge = constraint_terms(model, compact, local)
            weight = stop - start
            cross_sum += float(cross.cpu()) * weight
            monitor_sum += float(monitor.cpu()) * weight
            bridge_sum += float(bridge.cpu()) * weight
            count += weight
    ce_value = ce_sum / len(groups)
    direct_value = direct_sum / len(groups)
    return {
        "ce": ce_value, "direct": direct_value,
        "norm_cross": cross_sum / count, "norm_monitor": monitor_sum / count,
        "bridge": bridge_sum / count,
        "total": ce_value + LAMBDA_DIRECT * (direct_value - math.log(2.0))
            + LAMBDA_NORMALIZATION * (cross_sum / count)
            + LAMBDA_BRIDGE * (bridge_sum / count),
    }

def learning_rate_for_epoch(epoch, n_epochs):
    fraction = (epoch + 1) / max(1, n_epochs)
    if fraction <= 0.25: return 1.0e-3
    if fraction <= 0.50: return 3.0e-4
    if fraction <= 0.72: return 1.0e-4
    if fraction <= 0.88: return 3.0e-5
    return 1.0e-5

def train_structured_ensemble(
    train_groups, validation_groups, train_bundle, validation_bundle,
    checkpoint_dir, seed,
):
    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    flat = train_groups.reshape(-1, train_groups.shape[-1])
    center, scale = fit_classifier_transform(flat)
    train_tensor = torch.as_tensor(
        transform_classifier_points(train_groups, center, scale), dtype=torch.float32
    )
    validation_tensor = torch.as_tensor(
        transform_classifier_points(validation_groups, center, scale), dtype=torch.float32
    )
    train_constraints = prepare_constraint_bundle(train_bundle, center, scale)
    validation_constraints = prepare_constraint_bundle(validation_bundle, center, scale)
    config = {
        "input_dim": int(train_groups.shape[-1]),
        "width": int(campaign["class_width"]), "blocks": int(campaign["class_blocks"]),
        "log_ratio_bound": LOG_RATIO_BOUND, "epochs": campaign["class_epochs"],
        "lambda_direct": LAMBDA_DIRECT, "lambda_norm": LAMBDA_NORMALIZATION,
        "lambda_bridge": LAMBDA_BRIDGE, "objective": CAMPAIGN_SCHEMA,
        "campaign_signature": CAMPAIGN_SIGNATURE, "campaign_seed": SEED,
        "bridge_component_allocation": "iid",
        "input_transform": "robust_median_mad_asinh_v1",
    }
    fingerprint = classifier_fingerprint(
        train_groups, validation_groups, train_bundle, validation_bundle, config
    )
    members = []
    for member in range(campaign["flow_members"]):
        checkpoint = checkpoint_dir / f"member_{member:02d}.pt"
        if LOAD_IF_AVAILABLE and checkpoint.exists():
            saved = safe_torch_load(checkpoint)
            if saved.get("fingerprint") != fingerprint:
                raise RuntimeError(f"Classifier checkpoint fingerprint mismatch: {checkpoint}")
            model = StructuredThreeClass(**saved["config_model"]).to(device)
            model.load_state_dict(saved["state_dict"])
            model.eval()
            if saved.get("input_transform") != "robust_median_mad_asinh_v1":
                raise RuntimeError(f"Classifier checkpoint uses a stale input transform: {checkpoint}")
            members.append({
                "model": model, "center": center, "scale": scale,
                "history": saved["history"],
            })
            print("Loaded", checkpoint)
            continue

        member_seed = int(seed + 10_007 * member)
        seed_everything(member_seed)
        model = StructuredThreeClass(
            config["input_dim"], config["width"], config["blocks"], config["log_ratio_bound"]
        ).to(device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=1.0e-3, weight_decay=1.0e-5)
        generator = torch.Generator().manual_seed(member_seed + 1)
        loader = DataLoader(
            TensorDataset(train_tensor), batch_size=512 if PROFILE != "SMOKE" else 128,
            shuffle=True, generator=generator,
        )
        constraint_rng = np.random.default_rng(member_seed + 2)
        history = {k: [] for k in [
            "train_ce", "train_direct", "train_norm_cross", "train_bridge",
            "validation_ce", "validation_direct", "validation_norm_cross",
            "validation_norm", "validation_bridge", "validation_total",
            "normalization_scale", "bridge_scale", "checkpoint_eligible",
            "learning_rate",
        ]}
        best_state, best_value, stale = None, math.inf, 0
        n_epochs = int(campaign["class_epochs"])
        for epoch in range(n_epochs):
            lr = learning_rate_for_epoch(epoch, n_epochs)
            for group in optimizer.param_groups:
                group["lr"] = lr
            normalization_scale = min(
                1.0,
                max(0.0, (epoch + 1 - 0.12 * n_epochs) / max(1.0, 0.20 * n_epochs)),
            )
            bridge_scale = min(
                1.0,
                max(0.0, (epoch + 1 - 0.38 * n_epochs) / max(1.0, 0.22 * n_epochs)),
            )
            running = np.zeros(4, dtype=np.float64)
            rows = 0
            model.train()
            for (group_batch,) in loader:
                optimizer.zero_grad(set_to_none=True)
                ce, direct = class_losses(model, group_batch)
                n_constraint = min(
                    int(campaign["constraint_batch"]),
                    len(train_constraints["zp_a"]), len(train_constraints["bridge"]),
                )
                index_norm = constraint_rng.choice(len(train_constraints["zp_a"]), n_constraint, replace=False)
                index_bridge = constraint_rng.choice(len(train_constraints["bridge"]), n_constraint, replace=False)
                compact = {
                    "zp_a": train_constraints["zp_a"][index_norm],
                    "zp_b": train_constraints["zp_b"][index_norm],
                    "zl_a": train_constraints["zl_a"][index_norm],
                    "zl_b": train_constraints["zl_b"][index_norm],
                    "bridge": train_constraints["bridge"][index_bridge],
                    "bridge_base": train_constraints["bridge_base"][index_bridge],
                }
                local = torch.arange(n_constraint)
                norm_cross, _, bridge = constraint_terms(model, compact, local)
                loss = (
                    ce + LAMBDA_DIRECT * (direct - math.log(2.0))
                    + normalization_scale * LAMBDA_NORMALIZATION * norm_cross
                    + bridge_scale * LAMBDA_BRIDGE * bridge
                )
                if not bool(torch.isfinite(loss)):
                    raise FloatingPointError("Non-finite classifier objective")
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                optimizer.step()
                weight = len(group_batch)
                running += weight * np.array([
                    float(ce.detach().cpu()), float(direct.detach().cpu()),
                    float(norm_cross.detach().cpu()), float(bridge.detach().cpu()),
                ])
                rows += weight
            validation = validation_objective(model, validation_tensor, validation_constraints)
            history["train_ce"].append(running[0] / rows)
            history["train_direct"].append(running[1] / rows)
            history["train_norm_cross"].append(running[2] / rows)
            history["train_bridge"].append(running[3] / rows)
            history["validation_ce"].append(validation["ce"])
            history["validation_direct"].append(validation["direct"])
            history["validation_norm_cross"].append(validation["norm_cross"])
            history["validation_norm"].append(validation["norm_monitor"])
            history["validation_bridge"].append(validation["bridge"])
            history["validation_total"].append(validation["total"])
            history["normalization_scale"].append(normalization_scale)
            history["bridge_scale"].append(bridge_scale)
            checkpoint_eligible = (normalization_scale >= 1.0 and bridge_scale >= 1.0)
            history["checkpoint_eligible"].append(checkpoint_eligible)
            history["learning_rate"].append(lr)
            if checkpoint_eligible and validation["total"] < best_value:
                best_value = validation["total"]
                best_state = copy.deepcopy(model.state_dict())
                stale = 0
            elif checkpoint_eligible:
                stale += 1
            print(
                f"classifier {member + 1}/{campaign['flow_members']} epoch {epoch + 1:03d}: "
                f"CE={history['train_ce'][-1]:.4f}, val={validation['total']:.4f}, "
                f"Zcross={validation['norm_cross']:.3g}, Zmon={validation['norm_monitor']:.3g}, "
                f"bridge={validation['bridge']:.3g}, scales=({normalization_scale:.2f},{bridge_scale:.2f}), lr={lr:.1e}"
            )
            if (
                checkpoint_eligible
                and epoch > 0.88 * n_epochs
                and stale >= max(4, n_epochs // 8)
            ):
                break
        if best_state is None:
            raise RuntimeError("Classifier training never produced a finite checkpoint")
        model.load_state_dict(best_state)
        model.eval()
        torch.save({
            "state_dict": best_state,
            "config_model": {
                "input_dim": config["input_dim"], "width": config["width"],
                "blocks": config["blocks"], "log_ratio_bound": config["log_ratio_bound"],
            },
            "config": config, "center": center, "scale": scale,
            "input_transform": "robust_median_mad_asinh_v1",
            "history": history, "fingerprint": fingerprint,
        }, checkpoint)
        members.append({
            "model": model, "center": center, "scale": scale,
            "history": history,
        })
    return members, validation_constraints


In [ ]:
def predict_heads(ensemble, points, batch_size=16_384):
    points = np.asarray(points, dtype=np.float32)
    sums_a = np.zeros(len(points), dtype=np.float64)
    sums_c = np.zeros(len(points), dtype=np.float64)
    for pack in ensemble:
        standardized = transform_classifier_points(
            points, pack["center"], pack["scale"]
        )
        chunks_a, chunks_c = [], []
        pack["model"].eval()
        with torch.no_grad():
            for start in range(0, len(points), batch_size):
                tensor = torch.as_tensor(standardized[start:start + batch_size], device=device)
                a, c = pack["model"].heads(tensor)
                chunks_a.append(a.cpu().numpy())
                chunks_c.append(c.cpu().numpy())
        sums_a += np.concatenate(chunks_a)
        sums_c += np.concatenate(chunks_c)
    return sums_a / len(ensemble), sums_c / len(ensemble)

def deployed_ensemble_constraint_diagnostics(ensemble, bundle):
    """Held-out constraints of the actually deployed averaged a/c heads."""
    def heads(name):
        values = np.asarray(bundle[name], dtype=np.float32)
        shape = values.shape[:-1]
        a, c = predict_heads(ensemble, values.reshape(-1, values.shape[-1]))
        return a.reshape(shape), c.reshape(shape)

    a_zp_a, _ = heads("zp_a")
    a_zp_b, _ = heads("zp_b")
    _, c_zl_a = heads("zl_a")
    _, c_zl_b = heads("zl_b")
    log_zp_a = logsumexp(a_zp_a, axis=1) - math.log(a_zp_a.shape[1])
    log_zp_b = logsumexp(a_zp_b, axis=1) - math.log(a_zp_b.shape[1])
    log_zl_a = logsumexp(c_zl_a, axis=1) - math.log(c_zl_a.shape[1])
    log_zl_b = logsumexp(c_zl_b, axis=1) - math.log(c_zl_b.shape[1])
    dzp_a, dzp_b = np.expm1(log_zp_a), np.expm1(log_zp_b)
    dzl_a, dzl_b = np.expm1(log_zl_a), np.expm1(log_zl_b)
    cross = 0.5 * (np.mean(dzp_a * dzp_b) + np.mean(dzl_a * dzl_b))
    monitor = 0.5 * (
        np.mean(np.square(0.5 * (dzp_a + dzp_b)))
        + np.mean(np.square(0.5 * (dzl_a + dzl_b)))
    )
    a_bridge, c_bridge = heads("bridge")
    raw_evidence = np.asarray(bundle["bridge_base"], dtype=np.float64) + c_bridge - a_bridge
    if raw_evidence.shape[1] < 2:
        raise ValueError("Deployed bridge diagnostic needs at least two IID candidates.")
    bridge = np.var(raw_evidence, axis=1, ddof=1).mean()
    result = {
        "norm_cross": float(cross), "norm_monitor": float(monitor),
        "bridge": float(bridge),
    }
    if not np.isfinite(list(result.values())).all():
        raise FloatingPointError("Non-finite deployed-ensemble constraint diagnostic")
    return result

def normalized_weights(log_values):
    log_values = np.asarray(log_values, dtype=np.float64)
    weights = np.exp(log_values - logsumexp(log_values))
    if not np.isfinite(weights).all() or weights.sum() <= 0:
        raise FloatingPointError("Invalid importance weights")
    return weights / weights.sum()

def effective_sample_size(weights):
    weights = np.asarray(weights, dtype=np.float64)
    return float(1.0 / np.square(weights).sum())

def draw_fold_posterior(
    task, transform, observation, widths, q_phi, q_eta, classifier,
    n_proposal, n_samples, seed,
):
    discrete = str(task.name) in DISCRETE_TASKS
    n_jitter = int(campaign["observation_jitters"]) if discrete else 1
    x_context = np.repeat(np.asarray(observation, dtype=np.float32).reshape(1, -1), n_jitter, axis=0)
    if discrete:
        x_context = dequantize(x_context, widths, seed + 1)
    draws_per_context = max(256, int(math.ceil(n_proposal / n_jitter)))
    z_draws = draw_flow_mixture(q_phi, draws_per_context, x_context, seed + 2)
    z_flat = z_draws.reshape(-1, z_draws.shape[-1])
    x_flat = np.repeat(x_context[:, None, :], draws_per_context, axis=1).reshape(-1, x_context.shape[-1])
    points = np.column_stack([z_flat, x_flat])
    a, c = predict_heads(classifier, points)
    log_qp = spline_flow_ensemble_log_prob(q_phi, z_flat, context=x_flat)
    log_ql = spline_flow_ensemble_log_prob(q_eta, x_flat, context=z_flat)
    raw_evidence = prior_log_prob_latent(task, transform, z_flat) + log_ql - log_qp + c - a
    a = a.reshape(n_jitter, draws_per_context)
    raw_evidence = raw_evidence.reshape(n_jitter, draws_per_context)

    context_log_evidence = np.median(raw_evidence, axis=1)
    context_weights = normalized_weights(context_log_evidence) if discrete else np.ones(1)
    all_log_weights = []
    for index in range(n_jitter):
        local = a[index] - logsumexp(a[index])
        all_log_weights.append(local + math.log(context_weights[index]))
    weights = normalized_weights(np.concatenate(all_log_weights))
    theta = transform.inverse(z_flat)
    rng = np.random.default_rng(int(seed + 3))
    selected = rng.choice(len(theta), size=int(n_samples), replace=True, p=weights)
    diagnostics = {
        "ESS": effective_sample_size(weights),
        "ESS_fraction": effective_sample_size(weights) / len(weights),
        "max_weight": float(weights.max()),
        "rms_log_ZP": float(np.sqrt(np.mean(np.square(logsumexp(a, axis=1) - math.log(draws_per_context))))),
        "bridge_sd_median": float(np.median(np.std(raw_evidence, axis=1))),
        "observation_jitters": n_jitter,
    }
    return theta[selected].astype(np.float32), diagnostics


def draw_fold_predictive(
    transform, posterior_theta, q_eta, classifier,
    n_outputs, n_candidates, seed,
):
    """Finite-K SIR draw from q_eta(x|z) exp(c(z,x)) for one fold."""
    posterior_theta = np.asarray(posterior_theta, dtype=np.float32)
    n_outputs = int(n_outputs)
    n_candidates = int(n_candidates)
    if n_outputs < 1 or n_candidates < 2:
        raise ValueError("Predictive SIR needs positive outputs and at least two candidates.")
    rng = np.random.default_rng(int(seed))
    chosen_theta = posterior_theta[
        rng.choice(len(posterior_theta), size=n_outputs, replace=n_outputs > len(posterior_theta))
    ]
    z = transform.forward(chosen_theta)
    x_candidates = draw_flow_mixture(
        q_eta, n_candidates, z, seed + 1, allocation="iid"
    )
    z_repeat = np.repeat(z[:, None, :], n_candidates, axis=1)
    points = np.concatenate([z_repeat, x_candidates], axis=2).reshape(
        -1, z.shape[1] + x_candidates.shape[2]
    )
    _, c = predict_heads(classifier, points)
    c = c.reshape(n_outputs, n_candidates)
    log_weights = c - logsumexp(c, axis=1, keepdims=True)
    weights = np.exp(log_weights)
    cdf = np.cumsum(weights, axis=1)
    uniforms = rng.random(n_outputs)
    selected = np.minimum(
        np.sum(cdf < uniforms[:, None], axis=1), n_candidates - 1
    )
    x_selected = x_candidates[np.arange(n_outputs), selected]
    candidate_ess = 1.0 / np.square(weights).sum(axis=1)
    diagnostics = {
        "candidate_ESS_mean": float(candidate_ess.mean()),
        "candidate_ESS_fraction_mean": float((candidate_ess / n_candidates).mean()),
        "candidate_ESS_fraction_min_over_theta": float((candidate_ess / n_candidates).min()),
        "candidate_max_weight_mean": float(weights.max(axis=1).mean()),
        "candidate_max_weight_worst": float(weights.max()),
        "candidate_log_ZL_rms": float(np.sqrt(np.mean(np.square(
            logsumexp(c, axis=1) - math.log(n_candidates)
        )))),
        "candidate_count": n_candidates,
        "outputs": n_outputs,
    }
    return chosen_theta.astype(np.float32), x_selected.astype(np.float32), diagnostics

def mmd_rbf(reference, candidate, seed, max_samples=2_000):
    reference = np.asarray(reference, dtype=np.float64)
    candidate = np.asarray(candidate, dtype=np.float64)
    rng = np.random.default_rng(int(seed))
    n = min(len(reference), len(candidate), int(max_samples))
    x = reference[rng.choice(len(reference), n, replace=False)]
    y = candidate[rng.choice(len(candidate), n, replace=False)]
    mean = reference.mean(axis=0)
    std = reference.std(axis=0)
    std = np.where(std > 1.0e-8, std, 1.0)
    x, y = (x - mean) / std, (y - mean) / std
    pilot = np.concatenate([x[: min(500, n)], y[: min(500, n)]])
    distances = pdist(pilot, metric="sqeuclidean")
    distances = distances[distances > 0]
    bandwidth2 = float(np.median(distances)) if len(distances) else 1.0
    bandwidth2 = max(bandwidth2, 1.0e-8)

    def kernel_mean(left, right, chunk=256):
        total = 0.0
        count = 0
        for start in range(0, len(left), chunk):
            d2 = cdist(left[start:start + chunk], right, metric="sqeuclidean")
            total += float(np.exp(-d2 / (2.0 * bandwidth2)).sum())
            count += d2.size
        return total / count

    value = kernel_mean(x, x) + kernel_mean(y, y) - 2.0 * kernel_mean(x, y)
    return max(0.0, float(value)), bandwidth2, n

def posterior_metrics(reference, candidate, seed):
    from sbibm.metrics import c2st
    rng = np.random.default_rng(int(seed))
    n = min(len(reference), len(candidate))
    reference_equal = np.asarray(reference)[rng.choice(len(reference), n, replace=False)]
    candidate_equal = np.asarray(candidate)[rng.choice(len(candidate), n, replace=False)]
    score = float(c2st(
        torch.as_tensor(reference_equal, dtype=torch.float32),
        torch.as_tensor(candidate_equal, dtype=torch.float32),
        seed=int(seed), n_folds=5, z_score=True,
    ).item())
    mmd2, bandwidth2, mmd_n = mmd_rbf(reference_equal, candidate_equal, seed + 1)
    return {
        "C2ST": score, "MMD2": mmd2, "MMD": math.sqrt(max(0.0, mmd2)),
        "MMD_bandwidth2": bandwidth2, "MMD_n": mmd_n,
    }


In [ ]:
def export_figure(fig, output_dir, script_name):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    path = export_standalone_figure_script(fig, script_name=script_name, output_dir=output_dir)
    fig.savefig(output_dir / f"{Path(script_name).stem}.png", dpi=220, bbox_inches="tight")
    fig.savefig(output_dir / f"{Path(script_name).stem}.pdf", bbox_inches="tight")
    print("Exported:", path)
    return path

def plot_posterior(reference, candidate, labels, output_dir, observation_number):
    reference = np.asarray(reference)
    candidate = np.asarray(candidate)
    rng = np.random.default_rng(SEED + observation_number)
    if reference.shape[1] == 2:
        fig, axes = plt.subplots(1, 3, figsize=(12.2, 3.7), constrained_layout=True)
        for ax, values, title, color in [
            (axes[0], reference, "official reference", "black"),
            (axes[1], candidate, "multiclass hybrid", "C1"),
        ]:
            index = rng.choice(len(values), min(3_000, len(values)), replace=False)
            ax.scatter(values[index, 0], values[index, 1], s=4, alpha=0.22, color=color, rasterized=True)
            ax.set(xlabel=labels[0], ylabel=labels[1], title=title)
        axes[2].hist(reference[:, 0], bins=60, density=True, histtype="step", lw=2, color="black", label="reference")
        axes[2].hist(candidate[:, 0], bins=60, density=True, histtype="step", lw=2, color="C1", label="hybrid")
        axes[2].set(xlabel=labels[0], ylabel="density", title="first marginal")
        axes[2].legend()
    else:
        n_show = min(4, reference.shape[1])
        fig, axes = plt.subplots(2, 2, figsize=(9.0, 7.0), constrained_layout=True)
        for index, ax in enumerate(axes.flat):
            if index >= n_show:
                ax.axis("off")
                continue
            ax.hist(reference[:, index], bins=50, density=True, histtype="step", lw=2, color="black", label="reference")
            ax.hist(candidate[:, index], bins=50, density=True, histtype="step", lw=2, color="C1", label="hybrid")
            ax.set(xlabel=labels[index], ylabel="density", title=f"marginal {index + 1}")
            ax.legend(fontsize=8)
    export_figure(fig, output_dir, f"posterior_observation_{observation_number}")
    plt.show()


def plot_predictive(
    reference_theta, reference_x, candidate_theta, candidate_x,
    output_dir, observation_number,
):
    reference_theta = np.asarray(reference_theta)
    reference_x = np.asarray(reference_x)
    candidate_theta = np.asarray(candidate_theta)
    candidate_x = np.asarray(candidate_x)
    fig, axes = plt.subplots(2, 2, figsize=(9.4, 7.2), constrained_layout=True)
    n_marginals = min(3, reference_x.shape[1])
    for index in range(3):
        ax = axes.flat[index]
        if index >= n_marginals:
            ax.axis("off")
            continue
        ax.hist(reference_x[:, index], bins=50, density=True, histtype="step", lw=2, color="black", label="official predictive")
        ax.hist(candidate_x[:, index], bins=50, density=True, histtype="step", lw=2, color="C2", label="dual hybrid predictive")
        ax.set(xlabel=fr"$x_{{{index + 1}}}$", ylabel="density", title=f"predictive marginal {index + 1}")
        ax.legend(fontsize=8)
    ax = axes.flat[3]
    rng = np.random.default_rng(SEED + 90_000 + int(observation_number))
    for theta, values, color, label in [
        (reference_theta, reference_x, "black", "official joint"),
        (candidate_theta, candidate_x, "C2", "dual hybrid joint"),
    ]:
        use = rng.choice(len(values), min(2_000, len(values)), replace=False)
        ax.scatter(theta[use, 0], values[use, 0], s=6, alpha=0.22, color=color, label=label, rasterized=True)
    ax.set(xlabel=r"$\theta_1$", ylabel=r"$x_1$", title="posterior-predictive joint slice")
    ax.legend(fontsize=8)
    export_figure(fig, output_dir, f"predictive_observation_{observation_number}")
    plt.show()

def plot_training_histories(histories, constraint_rows, output_dir):
    fig, axes = plt.subplots(1, 3, figsize=(13.5, 3.8), constrained_layout=True)
    for label, history in histories:
        axes[0].plot(history.get("validation_ce", []), alpha=0.75, label=label)
        axes[1].plot(history.get("validation_norm", []), alpha=0.75, label=label)
        axes[2].plot(history.get("validation_bridge", []), alpha=0.75, label=label)
    axes[0].set(title="held-out multiclass CE", xlabel="epoch", ylabel="CE")
    axes[1].set(title="held-out conditional mass", xlabel="epoch", ylabel="pooled squared error", yscale="log")
    axes[2].set(title="held-out raw bridge", xlabel="epoch", ylabel="evidence-path variance", yscale="log")
    for ax in axes:
        ax.grid(alpha=0.25)
    if len(histories) <= 10:
        axes[0].legend(fontsize=7)
    export_figure(fig, output_dir, "training_and_constraints")
    plt.show()
    return pd.DataFrame(constraint_rows)


## 4. Run one cached task (or an explicit list)

Each fold uses complementary rows: $(K-1)/K$ train both flow mixtures and the held-out $1/K$ trains/validates the structured discriminator and its constraints.  No fresh simulator calls are used for normalization, bridge banks, or the learned posterior predictive.  The prior is free to evaluate and the flow references are generative.

For integer observations, the posterior returned by each fold pools several jittered cells.  Within a cell, $a$ corrects $q_\phi$; across cells, the median raw bridge evidence supplies the relative marginal-density factor.  This is randomized quadrature of the declared dequantized model, not an exact discrete likelihood calculation.

While each fold's $q_\eta$ and classifier are still resident, the notebook generates a matched posterior predictive: it selects posterior $\theta$, proposes a profile-controlled $K$ IID candidates from that fold's $q_\eta(x\mid\theta)$ mixture, weights them by $e^{c(\theta,x)}$, and SIR-resamples one $x$.  Finite $K$ makes this an approximation; candidate ESS and maximum weight are recorded.

Only after every fold is trained do we load the official reference posterior.  A capped subset of those reference parameters is passed to the **official task simulator** to form the comparison predictive.  These calls are evaluation-only, are counted separately from the cached training budget, and cannot affect training or checkpoint selection.  Discrete simulator outputs receive exactly the same declared dequantization convention as the learned predictive.  If the official simulator/backend is unavailable at this stage, the task fails through the existing manifest—there is no fallback.


In [ ]:
def train_and_evaluate_task(task_name):
    task_index = ALL_TASKS.index(task_name)
    task_seed = SEED + 100_000 * task_index
    task = sbibm.get_task(task_name)
    cache_path = CACHE_ROOT / f"{task_name}_prior_predictive_{campaign['num_simulations']}_seed{task_seed}.npz"
    print(f"\n{task_name}: exact cache path is {cache_path}")
    try:
        bank = load_or_simulate_bank(
            task, campaign["num_simulations"], cache_path=cache_path,
            seed=task_seed, chunk_size=20_000,
        )
    except Exception as exc:
        if task_name in {"sir", "lotka_volterra"}:
            raise RuntimeError(
                f"Official {task_name} training-bank stage unavailable. Configure the official "
                f"sbibm Julia/diffeqtorch backend or place the exact cache at {cache_path}; "
                "the backend is still required later for official predictive simulation."
            ) from exc
        raise

    require_finite_rows(task_name, "training bank theta", bank.theta)
    require_finite_rows(task_name, "training bank x", bank.x)
    transform = infer_parameter_transform(task)
    z_all = require_finite_rows(
        task_name, "transformed training parameters", transform.forward(bank.theta)
    ).astype(np.float32)
    widths = dequantization_widths(task_name, bank.x)
    x_all = require_finite_rows(
        task_name, "dequantized training observations",
        dequantize(bank.x, widths, task_seed + 1),
    ).astype(np.float32)
    observations = {}
    for observation_number in campaign["observations"]:
        observation = task.get_observation(num_observation=int(observation_number))
        observation_array = observation.detach().cpu().numpy().reshape(1, -1).astype(np.float32)
        observations[int(observation_number)] = require_finite_rows(
            task_name, f"official observation {observation_number}", observation_array
        ).astype(np.float32)

    fold_draws = {number: [] for number in observations}
    fold_diagnostics = {number: [] for number in observations}
    fold_predictive_pairs = {number: [] for number in observations}
    fold_predictive_diagnostics = {number: [] for number in observations}
    histories = []
    constraint_rows = []
    for fold in range(campaign["n_folds"]):
        print("\n" + "=" * 90)
        print(f"{task_name}: cross-fit fold {fold + 1}/{campaign['n_folds']}")
        split = split_simulation_bank(
            bank, seed=task_seed + 17, fold=fold, n_folds=campaign["n_folds"]
        )
        flow_index = split["flow_indices"]
        ratio_index = split["ratio_indices"]
        rng = np.random.default_rng(task_seed + 100 + fold)
        order = rng.permutation(ratio_index)
        n_validation = max(24, int(round(0.18 * len(order))))
        validation_index = order[:n_validation]
        training_index = order[n_validation:]
        if np.intersect1d(flow_index, ratio_index).size or np.intersect1d(training_index, validation_index).size:
            raise RuntimeError("Cross-fit or classifier split leakage")

        fold_dir = MODEL_ROOT / task_name / RUN_TAG / f"fold_{fold:02d}"
        q_phi = train_spline_flow_ensemble(
            z_all[flow_index], context=x_all[flow_index],
            checkpoint=fold_dir / "q_phi.pt", ensemble_size=campaign["flow_members"],
            model_config=flow_model_config(z_all.shape[1]),
            training_config=FLOW_TRAINING_CONFIG, device=device,
            seed=task_seed + 1_000 + 100 * fold,
            load_if_available=LOAD_IF_AVAILABLE, verify_checkpoint_data=True,
        )
        q_eta = train_spline_flow_ensemble(
            x_all[flow_index], context=z_all[flow_index],
            checkpoint=fold_dir / "q_eta.pt", ensemble_size=campaign["flow_members"],
            model_config=flow_model_config(x_all.shape[1]),
            training_config=FLOW_TRAINING_CONFIG, device=device,
            seed=task_seed + 2_000 + 100 * fold,
            load_if_available=LOAD_IF_AVAILABLE, verify_checkpoint_data=True,
        )
        train_groups = build_class_groups(
            z_all[training_index], x_all[training_index], q_phi, q_eta,
            task_seed + 3_000 + 100 * fold,
        )
        validation_groups = build_class_groups(
            z_all[validation_index], x_all[validation_index], q_phi, q_eta,
            task_seed + 4_000 + 100 * fold,
        )
        train_bundle = build_constraint_bundle(
            task, transform, z_all[training_index], x_all[training_index], q_phi, q_eta,
            task_seed + 5_000 + 100 * fold,
        )
        validation_bundle = build_constraint_bundle(
            task, transform, z_all[validation_index], x_all[validation_index], q_phi, q_eta,
            task_seed + 6_000 + 100 * fold,
        )
        classifier, prepared_validation = train_structured_ensemble(
            train_groups, validation_groups, train_bundle, validation_bundle,
            fold_dir / "structured_classifier", task_seed + 7_000 + 100 * fold,
        )
        for member, pack in enumerate(classifier):
            histories.append((f"fold {fold + 1}, member {member + 1}", pack["history"]))
        member_validation = [
            validation_objective(pack["model"],
                torch.as_tensor(
                    transform_classifier_points(
                        validation_groups, pack["center"], pack["scale"]
                    ),
                    dtype=torch.float32,
                ),
                prepared_validation)
            for pack in classifier
        ]
        deployed_validation = deployed_ensemble_constraint_diagnostics(
            classifier, validation_bundle
        )
        constraint_rows.append({
            "task": task_name, "fold": fold,
            "validation_norm_cross": deployed_validation["norm_cross"],
            "validation_norm_monitor": deployed_validation["norm_monitor"],
            "validation_bridge": deployed_validation["bridge"],
            "member_mean_validation_norm_cross": float(np.mean([row["norm_cross"] for row in member_validation])),
            "member_mean_validation_norm_monitor": float(np.mean([row["norm_monitor"] for row in member_validation])),
            "member_mean_validation_bridge": float(np.mean([row["bridge"] for row in member_validation])),
            "member_mean_validation_CE": float(np.mean([row["ce"] for row in member_validation])),
        })

        for observation_number, observation in observations.items():
            draws, diagnostics = draw_fold_posterior(
                task, transform, observation, widths, q_phi, q_eta, classifier,
                campaign["n_proposal"], campaign["n_posterior"],
                task_seed + 10_000 + 1_000 * fold + observation_number,
            )
            fold_draws[observation_number].append(draws)
            diagnostics.update({"fold": fold, "num_observation": observation_number})
            fold_diagnostics[observation_number].append(diagnostics)
            n_predictive_fold = int(math.ceil(
                campaign["predictive_samples"] / campaign["n_folds"]
            ))
            predictive_theta, predictive_x, predictive_diagnostics = draw_fold_predictive(
                transform, draws, q_eta, classifier, n_predictive_fold,
                campaign["predictive_candidates"],
                task_seed + 15_000 + 1_000 * fold + observation_number,
            )
            fold_predictive_pairs[observation_number].append(
                (predictive_theta, predictive_x)
            )
            predictive_diagnostics.update({
                "fold": fold, "num_observation": observation_number
            })
            fold_predictive_diagnostics[observation_number].append(
                predictive_diagnostics
            )

        del q_phi, q_eta, classifier, train_groups, validation_groups
        del train_bundle, validation_bundle, prepared_validation
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    figure_dir = FIGURE_ROOT / task_name / RUN_TAG
    constraint_table = plot_training_histories(histories, constraint_rows, figure_dir)
    constraint_table.to_csv(RESULT_ROOT / f"{task_name}__{RUN_TAG}__constraints.csv", index=False)
    deployed_constraint_summary = {
        f"mean_deployed_{name}": float(constraint_table[name].mean())
        for name in ["validation_norm_cross", "validation_norm_monitor", "validation_bridge"]
    }
    predictive_diagnostic_table = pd.concat([
        pd.DataFrame(values) for values in fold_predictive_diagnostics.values()
    ], ignore_index=True)
    predictive_diagnostic_path = RESULT_ROOT / (
        f"{task_name}__{RUN_TAG}__predictive_diagnostics.csv"
    )
    predictive_diagnostic_table.to_csv(
        predictive_diagnostic_path, index=False
    )
    print("Saved predictive diagnostics:", predictive_diagnostic_path)

    rows = []
    labels = task.get_labels_parameters()
    for observation_number, chunks in fold_draws.items():
        pool = np.concatenate(chunks)
        rng = np.random.default_rng(task_seed + 20_000 + observation_number)
        candidate = pool[rng.choice(len(pool), campaign["n_posterior"], replace=False)]
        candidate = require_finite_rows(
            task_name, f"hybrid posterior observation {observation_number}", candidate
        ).astype(np.float32)
        reference = task.get_reference_posterior_samples(
            num_observation=int(observation_number)
        ).detach().cpu().numpy().astype(np.float32)
        reference = require_finite_rows(
            task_name, f"official posterior observation {observation_number}", reference
        ).astype(np.float32)
        posterior_result = posterior_metrics(
            reference, candidate, task_seed + 30_000 + observation_number
        )
        posterior_path = RESULT_ROOT / (
            f"{task_name}__{RUN_TAG}__observation{observation_number}"
            "__posterior_samples.npz"
        )
        np.savez_compressed(
            posterior_path, candidate_theta=candidate, reference_theta=reference,
            observation_number=np.asarray(observation_number),
            campaign_seed=np.asarray(SEED),
            campaign_signature=np.asarray(CAMPAIGN_SIGNATURE),
        )
        print("Saved posterior samples:", posterior_path)

        predictive_theta_pool = np.concatenate([
            pair[0] for pair in fold_predictive_pairs[observation_number]
        ])
        predictive_x_pool = np.concatenate([
            pair[1] for pair in fold_predictive_pairs[observation_number]
        ])
        n_predictive_eval = min(
            int(campaign["predictive_reference_calls"]),
            len(reference), len(predictive_theta_pool),
        )
        if n_predictive_eval < 32:
            raise RuntimeError("Too few posterior-predictive rows for a stable evaluation.")
        predictive_rng = np.random.default_rng(
            task_seed + 40_000 + observation_number
        )
        candidate_index = predictive_rng.choice(
            len(predictive_theta_pool), n_predictive_eval, replace=False
        )
        candidate_predictive_theta = predictive_theta_pool[candidate_index]
        candidate_predictive_x = predictive_x_pool[candidate_index]
        require_finite_rows(
            task_name, f"learned predictive theta observation {observation_number}",
            candidate_predictive_theta,
        )
        require_finite_rows(
            task_name, f"learned predictive x observation {observation_number}",
            candidate_predictive_x,
        )
        reference_index = predictive_rng.choice(
            len(reference), n_predictive_eval, replace=False
        )
        reference_predictive_theta = reference[reference_index]

        # Evaluation-only official simulator calls. The model and all
        # checkpoints were fixed before the reference posterior was accessed.
        seed_everything(task_seed + 50_000 + observation_number)
        simulator = task.get_simulator(max_calls=n_predictive_eval)
        reference_predictive_raw = simulator(
            torch.as_tensor(reference_predictive_theta, dtype=torch.float32)
        )
        require_finite_rows(
            task_name,
            f"official predictive raw observation {observation_number} ({n_predictive_eval} calls)",
            reference_predictive_raw,
        )
        reference_predictive_x = task.flatten_data(
            reference_predictive_raw
        ).detach().cpu().numpy().astype(np.float32)
        reference_predictive_x = require_finite_rows(
            task_name,
            f"official predictive flattened observation {observation_number} ({n_predictive_eval} calls)",
            reference_predictive_x,
        ).astype(np.float32)
        reference_predictive_x = dequantize(
            reference_predictive_x, widths,
            task_seed + 60_000 + observation_number,
        )
        reference_predictive_x = require_finite_rows(
            task_name,
            f"official predictive evaluation observation {observation_number} ({n_predictive_eval} calls)",
            reference_predictive_x,
        ).astype(np.float32)
        print(
            f"{task_name} observation {observation_number}: "
            f"{n_predictive_eval:,} evaluation-only simulator calls "
            f"(outside the {campaign['num_simulations']:,}-call training bank)."
        )

        predictive_x_result = posterior_metrics(
            reference_predictive_x, candidate_predictive_x,
            task_seed + 70_000 + observation_number,
        )
        reference_joint = np.column_stack([
            reference_predictive_theta, reference_predictive_x
        ])
        candidate_joint = np.column_stack([
            candidate_predictive_theta, candidate_predictive_x
        ])
        predictive_joint_result = posterior_metrics(
            reference_joint, candidate_joint,
            task_seed + 80_000 + observation_number,
        )
        diagnostics = pd.DataFrame(
            fold_diagnostics[observation_number]
        ).mean(numeric_only=True).to_dict()
        predictive_diagnostic_frame = pd.DataFrame(
            fold_predictive_diagnostics[observation_number]
        )
        predictive_diagnostics = predictive_diagnostic_frame.mean(
            numeric_only=True
        ).to_dict()

        predictive_path = RESULT_ROOT / (
            f"{task_name}__{RUN_TAG}__observation{observation_number}"
            "__predictive_samples.npz"
        )
        np.savez_compressed(
            predictive_path,
            candidate_theta=candidate_predictive_theta,
            candidate_x=candidate_predictive_x,
            reference_theta=reference_predictive_theta,
            reference_x=reference_predictive_x,
            dequantization_widths=widths,
            evaluation_simulator_calls=np.asarray(n_predictive_eval),
            predictive_candidates_per_theta=np.asarray(
                campaign["predictive_candidates"]
            ),
            predictive_candidate_allocation=np.asarray("iid"),
            campaign_seed=np.asarray(SEED),
            campaign_signature=np.asarray(CAMPAIGN_SIGNATURE),
        )
        print("Saved predictive samples:", predictive_path)
        plot_posterior(
            reference, candidate, labels, figure_dir, observation_number
        )
        plot_predictive(
            reference_predictive_theta, reference_predictive_x,
            candidate_predictive_theta, candidate_predictive_x,
            figure_dir, observation_number,
        )

        rows.append({
            "task": task_name, "profile": PROFILE, "run_tag": RUN_TAG,
            "seed": SEED, "campaign_schema": CAMPAIGN_SCHEMA,
            "campaign_signature": CAMPAIGN_SIGNATURE,
            "num_simulations": campaign["num_simulations"],
            "num_observation": observation_number,
            "data_semantics": "dequantized_continuous_density" if task_name in DISCRETE_TASKS else "continuous_density",
            "dequantization_exact_unit_cell": task_name in {"bernoulli_glm_raw", "sir"},
            "dequantization_width_min": float(widths.min()),
            "dequantization_width_max": float(widths.max()),
            "bridge_component_allocation": "iid",
            "predictive_method": "fold_local_q_eta_correction_finite_K_SIR",
            "predictive_candidate_allocation": "iid",
            "predictive_candidates_per_theta": campaign["predictive_candidates"],
            "predictive_evaluation_simulator_calls": n_predictive_eval,
            "predictive_simulator_calls_in_training": 0,
            "predictive_candidate_max_weight_worst": float(
                predictive_diagnostic_frame["candidate_max_weight_worst"].max()
            ),
            "predictive_candidate_ESS_fraction_min_over_theta": float(
                predictive_diagnostic_frame["candidate_ESS_fraction_min_over_theta"].min()
            ),
            "C2ST": posterior_result["C2ST"],
            "MMD2": posterior_result["MMD2"],
            "MMD": posterior_result["MMD"],
            **{f"posterior_{key}": value for key, value in posterior_result.items()},
            **{f"predictive_x_{key}": value for key, value in predictive_x_result.items()},
            **{f"predictive_joint_{key}": value for key, value in predictive_joint_result.items()},
            **{f"mean_{key}": value for key, value in diagnostics.items() if key not in {"fold", "num_observation"}},
            **{f"mean_predictive_{key}": value for key, value in predictive_diagnostics.items() if key not in {"fold", "num_observation"}},
            **deployed_constraint_summary,
        })
    result = pd.DataFrame(rows)
    result_path = RESULT_ROOT / f"{task_name}__{RUN_TAG}__metrics.csv"
    result.to_csv(result_path, index=False)
    print("Saved:", result_path)
    display(result.style.format(precision=4).hide(axis="index"))
    return result


In [ ]:
campaign_results = []
campaign_failures = []
for selected_task in TASKS_TO_RUN:
    status_path = RESULT_ROOT / f"{selected_task}__{RUN_TAG}__status.json"
    try:
        result = train_and_evaluate_task(selected_task)
        campaign_results.append(result)
        status = {
            "task": selected_task, "profile": PROFILE, "seed": SEED,
            "run_tag": RUN_TAG, "campaign_schema": CAMPAIGN_SCHEMA,
            "campaign_signature": CAMPAIGN_SIGNATURE,
            "status": "completed",
        }
    except Exception as exc:
        status = {
            "task": selected_task, "profile": PROFILE, "seed": SEED,
            "run_tag": RUN_TAG, "campaign_schema": CAMPAIGN_SCHEMA,
            "campaign_signature": CAMPAIGN_SIGNATURE,
            "status": "failed", "exception_type": type(exc).__name__,
            "message": str(exc), "traceback": traceback.format_exc(),
        }
        campaign_failures.append(status)
        print(status["traceback"])
    status_path.write_text(json.dumps(status, indent=2), encoding="utf-8")
    print("Wrote status manifest:", status_path)

if campaign_failures:
    failure_table = pd.DataFrame(campaign_failures).drop(columns="traceback")
    display(failure_table)
    if FAIL_ON_TASK_ERROR:
        raise RuntimeError(
            "One or more requested sbibm tasks failed. No task was silently skipped; "
            "see the displayed table and per-task status JSON."
        )


## 5. Ten-task aggregate: a JANA-Figure-5-style posterior/predictive view

This is a **style-level analogue**, not a numerical reproduction of JANA Figure 5.  Its headline figure now has the correct two-part structure: posterior MMD and joint posterior-predictive MMD.  Here “joint” is the paired $(\theta,x_{\rm rep})$ distribution: the learned side pairs each fold's hybrid posterior draw with its corrected $q_\eta$ SIR draw; the reference side pairs an official posterior draw with a fresh official-simulator draw.  We additionally store $x_{\rm rep}$-only predictive MMD and both posterior/predictive C2ST.

MMD uses reference z-scoring, a median-heuristic Gaussian kernel, a biased non-negative $\mathrm{MMD}^2$ estimator, and at most 2,000 samples.  CSV files retain both $\mathrm{MMD}^2$ and $\sqrt{\mathrm{MMD}^2}$.  Kernels, tasks, and budgets differ from JANA, so the figure supports a methodological comparison rather than point-by-point numerical reproduction.

By default the cell reads only files with the exact current `RUN_TAG`.  For a paper campaign, set `EX9B_AGGREGATE_SEEDS=29082026,29082027,...`; the aggregate then constructs the exact seed-specific tags under the current profile and configuration signature, validates every CSV/status manifest, and reports every missing task--seed pair.  It never globs across profiles or configurations.  A paper claim over “all `sbibm`” requires `complete=True`, all requested task--seed status manifests completed, FULL-profile results, healthy ESS/normalization/bridge diagnostics, and disclosure of the evaluation-only simulator calls.


In [ ]:
candidate_metric_files = sorted({
    path
    for aggregate_run_tag in AGGREGATE_RUN_TAGS
    for path in RESULT_ROOT.glob(f"*__{aggregate_run_tag}__metrics.csv")
})
tag_to_seed = dict(zip(AGGREGATE_RUN_TAGS, AGGREGATE_SEEDS))
metric_frames = []
for path in candidate_metric_files:
    task_from_path, tag_from_path, suffix = path.name.rsplit("__", 2)
    if suffix != "metrics.csv" or tag_from_path not in tag_to_seed:
        raise RuntimeError(f"Unexpected aggregate metric path: {path}")
    expected_seed = tag_to_seed[tag_from_path]
    status_path = RESULT_ROOT / f"{task_from_path}__{tag_from_path}__status.json"
    if not status_path.exists():
        print("Excluding metric without status manifest:", path)
        continue
    status = json.loads(status_path.read_text(encoding="utf-8"))
    expected_status = {
        "task": task_from_path, "profile": PROFILE,
        "seed": expected_seed, "run_tag": tag_from_path,
        "campaign_schema": CAMPAIGN_SCHEMA,
        "campaign_signature": CAMPAIGN_SIGNATURE,
    }
    mismatched_status = {
        key: (status.get(key), value)
        for key, value in expected_status.items() if status.get(key) != value
    }
    if mismatched_status:
        raise RuntimeError(
            f"Status/configuration mismatch for {status_path}: {mismatched_status}"
        )
    if status.get("status") != "completed":
        print("Excluding metric whose latest status is not completed:", path)
        continue
    frame = pd.read_csv(path)
    required = {
        "task", "profile", "seed", "run_tag", "campaign_schema",
        "campaign_signature",
        "num_simulations", "num_observation", "posterior_MMD",
        "predictive_joint_MMD", "posterior_C2ST", "predictive_joint_C2ST",
    }
    if frame.empty or not required.issubset(frame.columns):
        raise RuntimeError(f"Incomplete aggregate metric schema: {path}")
    csv_contract = {
        "task": {task_from_path}, "profile": {PROFILE},
        "seed": {int(expected_seed)}, "run_tag": {tag_from_path},
        "campaign_schema": {CAMPAIGN_SCHEMA},
        "campaign_signature": {CAMPAIGN_SIGNATURE},
        "num_simulations": {int(campaign['num_simulations'])},
    }
    observed_contract = {
        "task": set(frame["task"].astype(str)),
        "profile": set(frame["profile"].astype(str)),
        "seed": set(frame["seed"].astype(int)),
        "run_tag": set(frame["run_tag"].astype(str)),
        "campaign_schema": set(frame["campaign_schema"].astype(str)),
        "campaign_signature": set(frame["campaign_signature"].astype(str)),
        "num_simulations": set(frame["num_simulations"].astype(int)),
    }
    if observed_contract != csv_contract:
        raise RuntimeError(
            f"Refusing to mix an incompatible metric file {path}: "
            f"observed={observed_contract}, expected={csv_contract}"
        )
    expected_observations = set(map(int, campaign["observations"]))
    observed_observations = set(frame["num_observation"].astype(int))
    duplicated_observations = frame["num_observation"].astype(int).duplicated().any()
    if observed_observations != expected_observations or duplicated_observations:
        raise RuntimeError(
            f"Incomplete or duplicated observation rows in {path}: "
            f"observed={sorted(observed_observations)}, "
            f"expected={sorted(expected_observations)}, "
            f"duplicates={bool(duplicated_observations)}"
        )
    metric_frames.append(frame)
if not metric_frames:
    print("No completed metric files yet for", AGGREGATE_RUN_TAGS)
    aggregate_results = pd.DataFrame()
else:
    aggregate_results = pd.concat(metric_frames, ignore_index=True)
    aggregate_results = aggregate_results.loc[aggregate_results["task"].isin(ALL_TASKS)]
    key_columns = ["task", "seed", "num_observation"]
    if aggregate_results.duplicated(key_columns).any():
        duplicate_rows = aggregate_results.loc[
            aggregate_results.duplicated(key_columns, keep=False), key_columns
        ]
        raise RuntimeError(f"Duplicate aggregate rows detected:\n{duplicate_rows}")
    completed_pairs = set(zip(aggregate_results["task"], aggregate_results["seed"].astype(int)))
    expected_pairs = {(task, seed) for seed in AGGREGATE_SEEDS for task in ALL_TASKS}
    missing_pairs = sorted(expected_pairs - completed_pairs, key=lambda item: (item[1], item[0]))
    print("Completed task--seed pairs:", len(completed_pairs), "/", len(expected_pairs))
    print("Missing task--seed pairs (not silently averaged):", missing_pairs)
    print("complete=", not missing_pairs)
    summary = (
        aggregate_results.groupby("task", as_index=False)
        .agg(
            mean_posterior_MMD=("posterior_MMD", "mean"),
            std_posterior_MMD=("posterior_MMD", "std"),
            mean_predictive_x_MMD=("predictive_x_MMD", "mean"),
            mean_predictive_joint_MMD=("predictive_joint_MMD", "mean"),
            std_predictive_joint_MMD=("predictive_joint_MMD", "std"),
            mean_posterior_C2ST=("posterior_C2ST", "mean"),
            mean_predictive_joint_C2ST=("predictive_joint_C2ST", "mean"),
            evaluation_simulator_calls=("predictive_evaluation_simulator_calls", "sum"),
            observations=("num_observation", "nunique"),
            independent_seeds=("seed", "nunique"),
            metric_rows=("seed", "size"),
        )
    )
    aggregate_dir = FIGURE_ROOT / "aggregate" / AGGREGATE_TAG
    aggregate_dir.mkdir(parents=True, exist_ok=True)
    aggregate_results.to_csv(aggregate_dir / "all_tasks_metrics.csv", index=False)
    summary.to_csv(aggregate_dir / "all_tasks_summary.csv", index=False)
    (aggregate_dir / "aggregate_manifest.json").write_text(json.dumps({
        "profile": PROFILE, "campaign_schema": CAMPAIGN_SCHEMA,
        "campaign_signature": CAMPAIGN_SIGNATURE,
        "aggregate_seeds": AGGREGATE_SEEDS,
        "aggregate_run_tags": AGGREGATE_RUN_TAGS,
        "complete": not missing_pairs, "missing_task_seed_pairs": missing_pairs,
    }, indent=2), encoding="utf-8")
    display(summary.style.format(precision=4).hide(axis="index"))

    x_positions = np.arange(len(ALL_TASKS))
    short_labels = [
        "G-Lin", "G-Unif", "G-Mix", "Moons", "SLCP",
        "SLCP-D", "B-GLM", "B-Raw", "SIR", "Lotka",
    ]
    fig, axes = plt.subplots(1, 2, figsize=(14.5, 5.0), constrained_layout=True)
    for x_index, task_name in enumerate(ALL_TASKS):
        rows = aggregate_results.loc[aggregate_results["task"] == task_name]
        if rows.empty:
            for ax in axes:
                ax.scatter([x_index], [1.0e-6], marker="x", color="0.75", zorder=3)
            continue
        jitter = np.linspace(-0.12, 0.12, len(rows)) if len(rows) > 1 else np.zeros(1)
        for ax, column, color in [
            (axes[0], "posterior_MMD", "C1"),
            (axes[1], "predictive_joint_MMD", "C2"),
        ]:
            values = np.maximum(rows[column].to_numpy(), 1.0e-7)
            ax.scatter(x_index + jitter, values, s=24, color=color, alpha=0.6)
            ax.scatter([x_index], [max(rows[column].mean(), 1.0e-7)], s=65, color=color, edgecolor="black", zorder=4)
    axes[0].set(yscale="log", ylabel=r"Gaussian-kernel MMD", title="(a) Posterior MMD")
    axes[1].set(yscale="log", ylabel=r"Gaussian-kernel MMD", title="(b) Joint posterior-predictive MMD")
    for ax in axes:
        ax.set_xticks(x_positions, short_labels, rotation=35, ha="right")
        ax.grid(axis="y", alpha=0.25)
    export_figure(fig, aggregate_dir, "all_tasks_jana_style_posterior_predictive_mmd")
    plt.show()

    fig, axes = plt.subplots(1, 2, figsize=(14.5, 5.0), constrained_layout=True)
    for x_index, task_name in enumerate(ALL_TASKS):
        rows = aggregate_results.loc[aggregate_results["task"] == task_name]
        if rows.empty:
            for ax in axes:
                ax.scatter([x_index], [0.5], marker="x", color="0.75", zorder=3)
            continue
        jitter = np.linspace(-0.12, 0.12, len(rows)) if len(rows) > 1 else np.zeros(1)
        for ax, column, color in [
            (axes[0], "posterior_C2ST", "C0"),
            (axes[1], "predictive_joint_C2ST", "C4"),
        ]:
            ax.scatter(x_index + jitter, rows[column], s=24, color=color, alpha=0.6)
            ax.scatter([x_index], [rows[column].mean()], s=65, color=color, edgecolor="black", zorder=4)
    axes[0].set(ylabel="C2ST", title="(a) Posterior C2ST")
    axes[1].set(ylabel="C2ST", title="(b) Joint predictive C2ST")
    for ax in axes:
        ax.axhline(0.5, color="black", ls="--", lw=1)
        ax.set_xticks(x_positions, short_labels, rotation=35, ha="right")
        ax.grid(axis="y", alpha=0.25)
    export_figure(fig, aggregate_dir, "all_tasks_supplemental_c2st")
    plt.show()


## Conclusions and paper-run checklist

A completed run tests one precise claim: a shared structured discriminator, trained with CE, a direct posterior-ratio objective, unbiased cross-normalization, and raw evidence invariance, can correct two learned flow-mixture references for both posterior inference and simulator-free posterior-predictive generation.

Before using the aggregate in a paper:

- run the identical FULL profile for all ten tasks with explicit `EX9B_SEED` values, then aggregate only those seeds through `EX9B_AGGREGATE_SEEDS`;
- treat every prior-predictive training bank as the recorded training simulator budget and combine only seed-specific `RUN_TAG`s sharing the exact profile and `CAMPAIGN_SIGNATURE`;
- report the separately capped official-simulator calls used only to construct the reference posterior predictive;
- require healthy held-out conditional-mass and bridge diagnostics together with posterior and predictive MMD/C2ST;
- report posterior importance ESS and predictive candidate ESS/maximum weights; finite-$K$ SIR failure must not be hidden by a favorable aggregate score;
- keep the dequantized likelihood language only for the two Bernoulli tasks and SIR, do not call the `bernoulli_glm` smoothed density an exact PMF, and treat Lotka--Volterra as continuous;
- disclose the memory-aware flow configuration used for 100-dimensional observations and the profile-specific predictive $K$;
- configure the official ODE backends for SIR/Lotka--Volterra; an exact cached bank avoids training calls but does not replace task/reference access or the evaluation-only predictive simulator, whose failure is a task failure rather than a silent omission;
- describe the two-panel plot as JANA-Figure-5-style: its kernels, benchmark tasks, simulation budgets, and finite-$K$ construction are not an exact numerical reproduction;
- remember that the official reference posterior and predictive simulator calls enter only after training, but tuning after viewing them would still be test-set adaptation.
